<a href="https://colab.research.google.com/github/edwincas4240-ctrl/inteligencia-artificial-2/blob/main/modelo_bogota_prediccion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏡 Predicción y Clasificación de Precios Inmobiliarios en Bogotá, Colombia
¡Hola! En este notebook vamos a simular y analizar un dataset de predios e inmuebles adaptado específicamente a **Bogotá, Colombia**.
Compararemos el rendimiento de tres modelos de Machine Learning:
1. **Regresión Lineal** (para capturar tendencias globales).
2. **Regresión Polinómica** (para modelar curvas no lineales complejas en la ciudad).
3. **Regresión Logística** (para clasificar si una vivienda es de 'Alto Costo' o 'Económica').

Cada línea de código cuenta con un comentario personal detallado y reflexivo explicando su propósito exacto.

In [ ]:
# Importamos numpy para realizar operaciones matemáticas y manipulación eficiente de matrices numéricas.
import numpy as np
# Importamos pandas para gestionar nuestros datos estructurados en formato tabular (DataFrames).
import pandas as pd
# Importamos pyplot de matplotlib para generar gráficos estadísticos y visualizar el comportamiento de los modelos.
import matplotlib.pyplot as plt
# Importamos seaborn para crear gráficos más estéticos y profesionales de manera sencilla.
import seaborn as sns
# Configuramos el entorno para que los gráficos generados se muestren directamente dentro del notebook.
%matplotlib inline

## 1. Simulación y Carga de Datos Inmobiliarios de Bogotá
Como no tenemos un archivo externo preconfigurado, generaremos un dataset sintético realista basado en las características del mercado inmobiliario bogotano (áreas en $m^2$, estratos socioeconómicos del 1 al 6, habitaciones, cercanía a TransMilenio y precio en millones de pesos colombianos - COP).

In [ ]:
# Fijamos una semilla aleatoria (random_state) para asegurar que los datos generados sean exactamente iguales en cada ejecución.
np.random.seed(42)
# Definimos el número total de predios bogotanos que vamos a simular para nuestro estudio.
n_predios = 1000
# Generamos áreas aleatorias para las propiedades en metros cuadrados, entre 40 m² y 250 m².
area_m2 = np.random.uniform(40, 250, n_predios)
# Simulamos los estratos socioeconómicos típicos en Bogotá eligiendo valores enteros aleatorios entre 1 y 6.
estrato = np.random.choice([1, 2, 3, 4, 5, 6], size=n_predios, p=[0.1, 0.2, 0.3, 0.2, 0.15, 0.05])
# Generamos el número de habitaciones de forma aleatoria entre 1 y 5.
habitaciones = np.random.choice([1, 2, 3, 4, 5], size=n_predios, p=[0.1, 0.3, 0.4, 0.15, 0.05])
# Simulamos la distancia en kilómetros a una troncal de TransMilenio, con valores entre 0.1 km y 8 km.
distancia_transmilenio = np.random.uniform(0.1, 8.0, n_predios)
# Calculamos el precio base en millones de pesos considerando un componente no lineal por estrato y área.
precio_millones = (area_m2 * 3.5) + (estrato * 70) + (habitaciones * 15) - (distancia_transmilenio * 10) + np.random.normal(0, 25, n_predios)
# Construimos un DataFrame consolidando todas las variables simuladas para la ciudad de Bogotá.
df_bogota = pd.DataFrame({
    'Area_m2': area_m2,
    'Estrato': estrato,
    'Habitaciones': habitaciones,
    'Distancia_TransMilenio_km': distancia_transmilenio,
    'Precio_Millones': precio_millones
})
# Mostramos las primeras 5 filas del dataset para verificar su estructura y valores lógicos.
df_bogota.head()

## 2. Análisis Exploratorio de Datos (EDA)
Revisamos las estadísticas descriptivas para confirmar que los precios y características guarden coherencia con la realidad inmobiliaria de Bogotá.

In [ ]:
# Solicitamos un resumen técnico general del DataFrame para comprobar tipos de datos y ausencia de nulos.
df_bogota.info()
# Calculamos métricas estadísticas clave (promedio, desviación estándar, mínimos y máximos) de las variables.
df_bogota.describe()

## 3. Preparación de los Datos para Modelos de Regresión
Definimos las variables independientes ($X$) y la variable objetivo continua ($y$, Precio en Millones).

In [ ]:
# Seleccionamos como características de entrada todas las columnas excepto el precio.
X = df_bogota.drop('Precio_Millones', axis=1)
# Definimos como variable objetivo (y) la columna del precio en millones de pesos.
y = df_bogota['Precio_Millones']
# Importamos la función train_test_split para dividir nuestros datos en conjuntos de entrenamiento y prueba.
from sklearn.model_selection import train_test_split
# Dividimos los datos: 80% para entrenar los modelos y 20% para probar su capacidad predictiva con datos nuevos.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 4. Modelo 1: Regresión Lineal
Evaluamos una relación lineal directa entre las características del predio y su precio estimado en Bogotá.

In [ ]:
# Importamos la clase LinearRegression de scikit-learn para crear nuestro modelo paramétrico lineal.
from sklearn.linear_model import LinearRegression
# Instanciamos el modelo de Regresión Lineal.
modelo_lr = LinearRegression()
# Entrenamos el modelo utilizando los datos de entrenamiento (X_train, y_train).
modelo_lr.fit(X_train, y_train)
# Generamos las predicciones del precio utilizando el conjunto de prueba (X_test).
y_pred_lr = modelo_lr.predict(X_test)

## 5. Modelo 2: Regresión Polinómica
Ampliamos el modelo lineal transformando las características en términos polinómicos (grado 2) para capturar curvaturas y efectos no lineales (ej. el impacto del estrato).

In [ ]:
# Importamos PolynomialFeatures para generar características polinómicas y de interacción entre variables.
from sklearn.preprocessing import PolynomialFeatures
# Instanciamos el generador de polinomios configurando un grado 2 y evitando términos de sesgo duplicados.
poly = PolynomialFeatures(degree=2, include_bias=False)
# Transformamos el conjunto de entrenamiento de características en formato polinómico.
X_train_poly = poly.fit_transform(X_train)
# Transformamos el conjunto de prueba utilizando las mismas reglas polinómicas aprendidas del entrenamiento.
X_test_poly = poly.transform(X_test)
# Instanciamos un nuevo modelo de Regresión Lineal para ajustarlo sobre los datos polinómicos transformados.
modelo_poly = LinearRegression()
# Entrenamos el modelo de regresión polinómica con las características expandidas.
modelo_poly.fit(X_train_poly, y_train)
# Generamos las predicciones del precio usando el conjunto de prueba polinómico.
y_pred_poly = modelo_poly.predict(X_test_poly)

## 6. Evaluación Comparativa de Modelos de Regresión
Calculamos métricas estándar (R² Score y RMSE) para comparar cuál modelo predice mejor el precio de las viviendas en Bogotá.

In [ ]:
# Importamos las métricas mean_squared_error y r2_score para evaluar cuantitativamente el error y el ajuste.
from sklearn.metrics import mean_squared_error, r2_score
# Calculamos el coeficiente de determinación R² para la Regresión Lineal.
r2_lr = r2_score(y_test, y_pred_lr)
# Calculamos la Raíz del Error Cuadrático Medio (RMSE) para la Regresión Lineal.
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
# Calculamos el coeficiente R² para la Regresión Polinómica.
r2_poly = r2_score(y_test, y_pred_poly)
# Calculamos el RMSE para la Regresión Polinómica.
rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))
# Agrupamos los resultados en un diccionario para estructurar nuestra tabla comparativa de regresión.
tabla_comparativa_reg = {
    'Modelo de Regresión': ['Regresión Lineal', 'Regresión Polinómica (Grado 2)'],
    'R2 Score (Ajuste)': [r2_lr, r2_poly],
    'RMSE (Error Promedio Millones COP)': [rmse_lr, rmse_poly]
}
# Convertimos el diccionario en un DataFrame de pandas para una visualización ordenada.
df_comp_reg = pd.DataFrame(tabla_comparativa_reg)
# Mostramos la tabla comparativa de los modelos de regresión.
print(df_comp_reg)

## 7. Modelo 3: Regresión Logística (Clasificación Inmobiliaria)
Convertimos nuestro problema en una tarea de clasificación binaria: queremos clasificar si un predio en Bogotá es considerado de **'Alto Costo'** (1) o **'Económico'** (0) tomando como umbral la mediana de los precios.

In [ ]:
# Calculamos la mediana del precio para usarla como punto de corte o umbral de clasificación.
mediana_precio = df_bogota['Precio_Millones'].median()
# Creamos una columna binaria objetivo (y_binaria): 1 si el precio es mayor a la mediana, 0 en caso contrario.
y_binaria = (df_bogota['Precio_Millones'] > mediana_precio).astype(int)
# Dividimos las características y la nueva etiqueta binaria en conjuntos de entrenamiento y prueba.
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_binaria, test_size=0.2, random_state=42)
# Importamos la clase LogisticRegression desde scikit-learn para clasificación binaria.
from sklearn.linear_model import LogisticRegression
# Instanciamos el modelo de Regresión Logística especificando max_iter para asegurar convergencia.
modelo_log = LogisticRegression(max_iter=1000)
# Entrenamos el modelo logístico utilizando los datos de entrenamiento de clasificación.
modelo_log.fit(X_train_c, y_train_c)
# Generamos las predicciones de clase (0 o 1) sobre el conjunto de prueba.
y_pred_log = modelo_log.predict(X_test_c)

## 8. Evaluación del Modelo de Clasificación (Regresión Logística)
Evaluamos el rendimiento de la Regresión Logística utilizando métricas de clasificación como el Accuracy (Precisión global) y la Matriz de Confusión.

In [ ]:
# Importamos accuracy_score y confusion_matrix para evaluar la calidad de las predicciones categóricas.
from sklearn.metrics import accuracy_score, confusion_matrix
# Calculamos la precisión global (accuracy) del modelo logístico en el conjunto de prueba.
acc_log = accuracy_score(y_test_c, y_pred_log)
# Generamos la matriz de confusión para ver aciertos y errores detallados por categoría.
matriz_conf = confusion_matrix(y_test_c, y_pred_log)
# Imprimimos los resultados de la clasificación.
print(f'Accuracy (Precisión Global) de la Regresión Logística: {acc_log:.4f}')
print('Matriz de Confusión:')
print(matriz_conf)

## 9. Conclusión y Análisis Personalizado para Bogotá

In [ ]:
# Imprimimos un análisis reflexivo adaptado al mercado inmobiliario de Bogotá, Colombia.
print('Análisis Personalizado - Inmobiliaria Bogotá:')
print('1. La Regresión Lineal ofrece una base sólida, pero ignora cómo el estrato y el área interactúan exponencialmente en sectores valorizados.')
print('2. La Regresión Polinómica mejora el ajuste al capturar curvas de precio más realistas según la estratificación bogotana.')
print('3. La Regresión Logística permite clasificar con alta efectividad si un predio pertenece al segmento de alto costo, facilitando decisiones comerciales.')